# 2000 Local Government Election Data Cleaning

This notebook prepares the 2000 local government election results for analysis. It loads the source data, standardizes column names, selects the required fields, converts text values into numeric types, removes excluded ballot records, aggregates party results, calculates municipality-level turnout, and exports the cleaned dataset.


## Import pandas

`pandas` supplies the DataFrame operations used throughout the cleaning workflow.

In [1]:
# Import pandas for tabular data cleaning and aggregation.
import pandas as pd
from pathlib import Path

# Paths are relative to the project folder, so this notebook runs on any computer.
# It works whether Jupyter is opened in the notebooks/ folder or in the project root.
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw"
CLEAN_DIR = PROJECT_DIR / "data" / "cleaned"

## Load the raw 2000 data

The source file uses `ISO-8859-1` encoding because this older dataset contains characters that may not decode correctly as UTF-8. Display the first ten rows for an initial review.

In [2]:
# Load the raw election results with the required source encoding.
df = pd.read_csv(RAW_DIR / "2000_LGE.csv.zip", encoding="ISO-8859-1")

# Inspect a small sample of the imported records.
df.head(10)

/tmp/ipykernel_136/1793977902.py:2: DtypeWarning: Columns (0: Spoilt
Votes) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RAW_DIR / "2000_LGE.csv.zip", encoding="ISO-8859-1")


,Electoral Event,Province,Municipality,Ward,Voting \nDistrict,Party,Ballot \nType,Registered\nVoters,% Voter \nTurnout,Total Votes \nCast,Valid Votes \nCast,Spoilt\nVotes
0,Local Government Elections 2000,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],24401001.0,11830014,AFRICAN NATIONAL CONGRESS,DC 40%,"1,107",66.49%,736,708,13
1,Local Government Elections 2000,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],24401001.0,11830014,AFRICAN NATIONAL CONGRESS,PR,"1,107",66.49%,734,706,9
2,Local Government Elections 2000,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],24401001.0,11830014,AFRICAN NATIONAL CONGRESS,WARD,"1,107",66.49%,735,699,13
3,Local Government Elections 2000,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],24401001.0,11830014,DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,PR,"1,107",66.49%,734,9,9
4,Local Government Elections 2000,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],24401001.0,11830014,DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,WARD,"1,107",66.49%,735,6,13
5,Local Government Elections 2000,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],24401001.0,11830014,INDEPENDENT,WARD,"1,107",66.49%,735,17,13
6,Local Government Elections 2000,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],24401001.0,11830014,INKATHA FREEDOM PARTY,DC 40%,"1,107",66.49%,736,10,13
7,Local Government Elections 2000,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],24401001.0,11830014,INKATHA FREEDOM PARTY,PR,"1,107",66.49%,734,6,9
8,Local Government Elections 2000,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],24401001.0,11830014,UNITED DEMOCRATIC MOVEMENT,DC 40%,"1,107",66.49%,736,5,13
9,Local Government Elections 2000,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],24401001.0,11830014,UNITED DEMOCRATIC MOVEMENT,PR,"1,107",66.49%,734,4,9


## Inspect the source columns

Review the original field names before cleaning them. This helps identify embedded newline characters and confirms the fields available in the source.

In [3]:
# Inspect the original column names before standardizing them.
df.columns

Index(['Electoral Event', 'Province', 'Municipality', 'Ward',
       'Voting \nDistrict', 'Party', 'Ballot \nType', 'Registered\nVoters',
       '% Voter \nTurnout', 'Total Votes \nCast', 'Valid Votes \nCast',
       'Spoilt\nVotes'],
      dtype='str')

## Clean column names

Remove newline characters embedded in the source labels so the columns can be referenced consistently in later operations.

In [4]:
# Remove newline characters from every column label.
df = df.rename(columns=lambda c: c.replace('\n', ''))

# Confirm the cleaned column names.
df.columns

Index(['Electoral Event', 'Province', 'Municipality', 'Ward',
       'Voting District', 'Party', 'Ballot Type', 'RegisteredVoters',
       '% Voter Turnout', 'Total Votes Cast', 'Valid Votes Cast',
       'SpoiltVotes'],
      dtype='str')

## Select the analysis fields

Keep only the province, municipality, party, ballot type, turnout, and valid-vote fields needed for the classification and prediction analysis. Other source fields are intentionally omitted from the working DataFrame.

In [5]:
# Keep only the fields needed for cleaning and aggregation.
draft = df.filter(axis=1, items=['Province','Municipality','Party','Ballot Type','% Voter Turnout','Valid Votes Cast'])

## Inspect the working DataFrame

Check the selected columns, data types, and non-null counts before converting the text-based numeric fields.

In [6]:
# Inspect the selected fields and their current data types.
draft.info()

<class 'pandas.DataFrame'>
RangeIndex: 199566 entries, 0 to 199565
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   Province          199566 non-null  str  
 1   Municipality      199566 non-null  str  
 2   Party             199563 non-null  str  
 3   Ballot Type       199566 non-null  str  
 4   % Voter Turnout   199566 non-null  str  
 5   Valid Votes Cast  199566 non-null  str  
dtypes: str(6)
memory usage: 9.1 MB


## Convert turnout and vote totals to numbers

Remove the percent sign from turnout values and thousands separators from vote totals, then convert both fields to numeric types suitable for calculations.

In [7]:
# Remove the percent sign and convert turnout to a numeric percentage.
draft['% Voter Turnout'] = draft['% Voter Turnout'].str.removesuffix('%').astype('float')

# Remove thousands separators and convert vote totals to integers.
draft['Valid Votes Cast'] = draft['Valid Votes Cast'].str.replace(',',"", regex=False).astype('int')

# Display the converted values before further cleaning.
draft

,Province,Municipality,Party,Ballot Type,% Voter Turnout,Valid Votes Cast
0,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],AFRICAN NATIONAL CONGRESS,DC 40%,66.49,708
1,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],AFRICAN NATIONAL CONGRESS,PR,66.49,706
2,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],AFRICAN NATIONAL CONGRESS,WARD,66.49,699
3,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,PR,66.49,9
4,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,WARD,66.49,6
...,...,...,...,...,...,...
199561,Western Cape,WCDMA05 [Central Karoo DC],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,DMA DC 60%,78.99,69
199562,Western Cape,WCDMA05 [Central Karoo DC],AFRICAN NATIONAL CONGRESS,DC 40%,69.06,34
199563,Western Cape,WCDMA05 [Central Karoo DC],AFRICAN NATIONAL CONGRESS,DMA DC 60%,69.06,36
199564,Western Cape,WCDMA05 [Central Karoo DC],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,DC 40%,69.06,85


## Check for duplicate rows

Inspect duplicate records before aggregation. The duplicate check is displayed but not assigned back to `draft`, so the working DataFrame remains unchanged.

In [8]:
# Inspect duplicates without assigning the result back to draft.
draft.drop_duplicates()

# Display the current working DataFrame.
draft

,Province,Municipality,Party,Ballot Type,% Voter Turnout,Valid Votes Cast
0,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],AFRICAN NATIONAL CONGRESS,DC 40%,66.49,708
1,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],AFRICAN NATIONAL CONGRESS,PR,66.49,706
2,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],AFRICAN NATIONAL CONGRESS,WARD,66.49,699
3,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,PR,66.49,9
4,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,WARD,66.49,6
...,...,...,...,...,...,...
199561,Western Cape,WCDMA05 [Central Karoo DC],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,DMA DC 60%,78.99,69
199562,Western Cape,WCDMA05 [Central Karoo DC],AFRICAN NATIONAL CONGRESS,DC 40%,69.06,34
199563,Western Cape,WCDMA05 [Central Karoo DC],AFRICAN NATIONAL CONGRESS,DMA DC 60%,69.06,36
199564,Western Cape,WCDMA05 [Central Karoo DC],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,DC 40%,69.06,85


## Exclude DC 40% records

Keep the PR and WARD ballot categories used in the analysis by removing `DC 40%` records.

In [9]:
# Exclude DC 40% ballot records from the analysis.
draft = draft[draft['Ballot Type']!="DC 40%"]

# Review the filtered records.
draft

,Province,Municipality,Party,Ballot Type,% Voter Turnout,Valid Votes Cast
1,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],AFRICAN NATIONAL CONGRESS,PR,66.49,706
2,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],AFRICAN NATIONAL CONGRESS,WARD,66.49,699
3,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,PR,66.49,9
4,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,WARD,66.49,6
5,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],INDEPENDENT,WARD,66.49,17
...,...,...,...,...,...,...
199557,Western Cape,WCDMA05 [Central Karoo DC],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,DMA DC 60%,65.61,109
199559,Western Cape,WCDMA05 [Central Karoo DC],AFRICAN NATIONAL CONGRESS,DMA DC 60%,78.99,36
199561,Western Cape,WCDMA05 [Central Karoo DC],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,DMA DC 60%,78.99,69
199563,Western Cape,WCDMA05 [Central Karoo DC],AFRICAN NATIONAL CONGRESS,DMA DC 60%,69.06,36


## Aggregate party results

Group records by province, municipality, party, and ballot type. Sum valid votes and average turnout across the ward-level records in each group.

In [10]:
# Aggregate vote totals and turnout by geography, party, and ballot type.
draft = (
    draft
    .groupby(['Province', 'Municipality','Party','Ballot Type'], as_index=False)
    .agg(
        ValidVotesCast=('Valid Votes Cast', 'sum'),
        MeanVoterTurnout=('% Voter Turnout', 'mean') # Turnout from each ward
    )
 )

# The filter above already removed DC 40% records; this repeat keeps the original logic explicit.
draft = draft[draft['Ballot Type']!="DC 40%"]

# Review the grouped records.
draft

,Province,Municipality,Party,Ballot Type,ValidVotesCast,MeanVoterTurnout
0,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],AFRICAN NATIONAL CONGRESS,PR,21333,53.354302
1,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],AFRICAN NATIONAL CONGRESS,WARD,20980,53.354302
2,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,PR,569,53.354302
3,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,WARD,19,56.350000
4,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],INDEPENDENT,WARD,1078,54.850000
...,...,...,...,...,...,...
2078,Western Cape,WCDMA04 [South Cape DC],COMMUNITY INITIATIVE/GEMEENSKAP INISIATIEF,DMA DC 60%,28,68.313333
2079,Western Cape,WCDMA04 [South Cape DC],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,DMA DC 60%,1875,68.313333
2080,Western Cape,WCDMA04 [South Cape DC],UNITED DEMOCRATIC MOVEMENT,DMA DC 60%,30,68.313333
2081,Western Cape,WCDMA05 [Central Karoo DC],AFRICAN NATIONAL CONGRESS,DMA DC 60%,1015,73.922000


## Calculate municipality-level turnout

Average the grouped turnout values within each municipality to create one municipality-level turnout measure, then round it to two decimal places.

In [11]:
# Average grouped turnout values to obtain one value per municipality.
municipality_turnout = (
    draft
    .groupby(['Municipality'], as_index=False)['MeanVoterTurnout']
    .mean()) # Turnout from each municipality

# Round municipality turnout percentages for consistent reporting.
municipality_turnout['MeanVoterTurnout'] = municipality_turnout['MeanVoterTurnout'].round(2)

# Review the municipality-level lookup table.
municipality_turnout

,Municipality,MeanVoterTurnout
0,CBDMA4 [Kruger Park],17.19
1,CBLC1 - Kuruman-Mothibistad [Kuruman],48.74
2,CBLC2 - Kungwini [Bronkhorstspruit],45.48
3,CBLC3 - Greater Marble Hall [Marble Hall],48.05
4,CBLC4 - Greater Groblersdal [Groblersdal],49.39
...,...,...
256,WCDMA01 [West Coast DC],71.05
257,WCDMA02 [Brede River DC],56.86
258,WCDMA03 [Overberg DC],39.17
259,WCDMA04 [South Cape DC],68.31


## Attach municipality turnout to each record

Remove the intermediate grouped turnout value and merge the municipality-level lookup back into the aggregated party records.

In [12]:
# Replace grouped turnout with the municipality-level mean turnout.
draft = draft.drop(columns=['MeanVoterTurnout']).merge(
    municipality_turnout,
    on='Municipality',
    how='left'
 )

# Display the final cleaned records before export.
draft

,Province,Municipality,Party,Ballot Type,ValidVotesCast,MeanVoterTurnout
0,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],AFRICAN NATIONAL CONGRESS,PR,21333,53.64
1,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],AFRICAN NATIONAL CONGRESS,WARD,20980,53.64
2,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,PR,569,53.64
3,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,WARD,19,53.64
4,Eastern Cape,EC05b1 - Umzimkulu [Umzimkulu],INDEPENDENT,WARD,1078,53.64
...,...,...,...,...,...,...
2078,Western Cape,WCDMA04 [South Cape DC],COMMUNITY INITIATIVE/GEMEENSKAP INISIATIEF,DMA DC 60%,28,68.31
2079,Western Cape,WCDMA04 [South Cape DC],DEMOCRATIC ALLIANCE/DEMOKRATIESE ALLIANSIE,DMA DC 60%,1875,68.31
2080,Western Cape,WCDMA04 [South Cape DC],UNITED DEMOCRATIC MOVEMENT,DMA DC 60%,30,68.31
2081,Western Cape,WCDMA05 [Central Karoo DC],AFRICAN NATIONAL CONGRESS,DMA DC 60%,1015,73.92


## Export the cleaned dataset

Save the completed 2000 election dataset as `data/cleaned/2000_LGE_Cleaned.csv` for downstream analysis.

In [13]:
# Export the cleaned 2000 election data for downstream analysis.
draft.to_csv(CLEAN_DIR / '2000_LGE_Cleaned.csv')